# Vehicle Tracking with YOLOv8

## Introduction

The objective of this mini-project is to try object tracking using a pretrained YOLOv8 model. I detect cars, buses, and trucks in a highway video and count their unique tracking IDs. This is an estimate based on tracker IDs, not necessarily the true number of vehicles, since a vehicle may receive a new ID if tracking is lost.


## Video and Model Preparation

I use `highway.mp4` as the input for this experiment and a pretrained YOLOv8 model for object tracking.


In [ ]:
# Install the Ultralytics package for YOLOv8
# The package includes pretrained detection models and tracking support
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00


In [ ]:
# Import YOLOv8
from ultralytics import YOLO

# OpenCV for reading and writing videos
import cv2

# For file and folder operations
import os

# To display videos in Google Colab
from IPython.display import Video, display

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Load model YOLOv8 small
model = YOLO("yolov8s.pt")

## Tracking and Counting

I limit detection to cars, buses, and trucks. The tracker assigns IDs to detected objects, and I collect the IDs in a set so repeated detections with the same ID are not counted again.


In [ ]:
# Input video path
video_path = "highway.mp4"

# Store unique vehicle tracking IDs in a set
# A set avoids duplicate IDs
unique_vehicle_ids = set()

# Run object tracking
# persist=True keeps tracking state across frames
results = model.track(
    source=video_path,
    conf=0.4,          # Confidence threshold
    iou=0.5,           # IoU threshold
    persist=True,      # Preserve tracking state
    classes=[2, 5, 7], # 2=car, 5=bus, 7=truck
    save=True          # Save the annotated video
)

# Collect tracking IDs from the output frames
for result in results:
    # Skip frames without tracking IDs
    if result.boxes.id is None:
        continue

    # Read all tracking IDs in the frame
    ids = result.boxes.id.cpu().numpy()

    # Add each tracking ID to the set
    for track_id in ids:
        unique_vehicle_ids.add(int(track_id))

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 217ms
Prepared 1 package in 88ms
Installed 1 package in 3ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/164) /content/highway.mp4: 384x640 1 car, 502.1ms
video 1/1 (frame 2/164) /content

In [ ]:
# Count unique tracking IDs
total_unique_vehicles = len(unique_vehicle_ids)

print("Unique tracking IDs detected:", total_unique_vehicles)

## Viewing the Result

I display the video with tracked objects after processing. The output location may vary with the Ultralytics version, so the file path may need to be updated.


In [ ]:
# Convert AVI to MP4 so the result can be
# displayed directly in Google Colab
def show_video(path):
    if path.endswith(".avi"):
        fixed = "highway_converted.mp4"
        !ffmpeg -y -i {path} -vcodec libx264 {fixed} -loglevel quiet
        display(Video(fixed, embed=True))
    else:
        display(Video(path, embed=True))

In [ ]:
# Ultralytics saves tracking results to an output directory
# often located under runs/detect
track_folder = sorted(os.listdir("runs/detect"))[-1]

# Output video path
output_video_path = f"runs/detect/{track_folder}/highway.avi"

# Display the output video
show_video(output_video_path)

## Conclusion

I explored how to track vehicles and count unique IDs with YOLOv8. I have not compared the tracked IDs against ground-truth vehicle counts, so this notebook remains a demonstration rather than a validated traffic-counting system.
